# 04 — Exploratory Data Analysis of the 1-Second SOLETE Data

## Purpose

The previous notebooks examined SOLETE at 60-minute, 5-minute, and 1-minute
resolution. We now inspect the 1-second file.

This file is much larger than the others, so the goal is **not** to load the
complete dataset into pandas. Instead, we use chunked HDF5 reads to answer
three questions:

1. Is the complete 1-second timestamp grid present and unique?
2. How are the measurements and candidate targets represented at this
   resolution?
3. Does the extra temporal detail justify the computational cost for later
   forecasting experiments?

No source value is modified in this notebook.

## Data source

SOLETE contains meteorological measurements together with co-located wind and
solar photovoltaic generation from the SYSLAB facility in Denmark.

- Pombo, D. V., Gehrke, O., & Bindner, H. W. (2022), *SOLETE, a 15-month
  long holistic dataset including: Meteorology, co-located wind and solar PV
  power from Denmark with various resolutions*, Data in Brief, 42, 108046.
  DOI: `10.1016/j.dib.2022.108046`
- Dataset DOI: `10.11583/DTU.17040767`


In [1]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from IPython.display import display

# Locate the project root whether VS Code starts from Project/ or notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATASET_PATH = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_1sec.h5"
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"SOLETE file not found: {DATASET_PATH}")

file_size_gb = DATASET_PATH.stat().st_size / (1024**3)

with h5py.File(DATASET_PATH, "r") as f:
    data = f["DATA"]
    columns = [name.decode("utf-8") for name in data["axis0"][:]]
    n_rows, n_columns = data["block0_values"].shape

file_summary = pd.Series({
    "file_size_GB": round(file_size_gb, 2),
    "rows": n_rows,
    "columns": n_columns,
}, name="1-second SOLETE")

display(file_summary.to_frame("value"))
print("Variables:")
for name in columns:
    print("-", name)


,value
file_size_GB,3.53
rows,39484801.00
columns,11.00


Variables:
- TEMPERATURE[degC]
- HUMIDITY[%]
- WIND_SPEED[m1s]
- WIND_DIR[deg]
- GHI[kW1m2]
- POA Irr[kW1m2]
- P_Gaia[kW]
- P_Solar[kW]
- Pressure[mbar]
- Azimuth[deg]
- Elevation[deg]


## Complete timestamp structure

A high-frequency forecasting dataset can only be used safely if its true time
axis is understood.

The HDF5 rows are not assumed to be chronological. We therefore scan the
timestamp array in chunks and determine:

- the true start and end times;
- whether observations lie on an exact one-second grid;
- whether timestamps are missing or duplicated;
- how often the stored row order jumps forward or backward; and
- the sizes of continuous blocks in the stored file.

The Boolean occupancy array used below represents the expected timestamp grid,
not the measurement matrix, so its memory cost is modest compared with loading
the complete 1-second dataset.


In [2]:
ONE_SECOND_NS = 1_000_000_000
CHUNK_SIZE = 1_000_000

# ---------- First pass: range, stored-order transitions, and block boundaries ----------
global_min = None
global_max = None
normal_transitions = 0
backward_jumps = 0
forward_jumps = 0
consecutive_duplicates = 0
boundary_starts = []
previous_last = None

with h5py.File(DATASET_PATH, "r") as f:
    timestamps_ds = f["DATA"]["axis1"]
    total_rows = len(timestamps_ds)

    for start in range(0, total_rows, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, total_rows)
        chunk = timestamps_ds[start:end]

        chunk_min = int(chunk.min())
        chunk_max = int(chunk.max())
        global_min = chunk_min if global_min is None else min(global_min, chunk_min)
        global_max = chunk_max if global_max is None else max(global_max, chunk_max)

        if previous_last is None:
            values = chunk
            diffs = np.diff(values)
            right_row_indices = start + np.arange(1, len(chunk))
        else:
            values = np.concatenate(([previous_last], chunk))
            diffs = np.diff(values)
            right_row_indices = start + np.arange(len(chunk))

        normal_transitions += int(np.sum(diffs == ONE_SECOND_NS))
        backward_jumps += int(np.sum(diffs < 0))
        forward_jumps += int(np.sum(diffs > ONE_SECOND_NS))
        consecutive_duplicates += int(np.sum(diffs == 0))

        breaks = np.where(diffs != ONE_SECOND_NS)[0]
        boundary_starts.extend(right_row_indices[breaks].tolist())

        previous_last = int(chunk[-1])

start_time = pd.to_datetime(global_min, unit="ns", utc=True)
end_time = pd.to_datetime(global_max, unit="ns", utc=True)

block_starts = np.array([0] + boundary_starts, dtype=np.int64)
block_ends = np.array(boundary_starts + [total_rows], dtype=np.int64)
block_lengths = block_ends - block_starts

# ---------- Second pass: global uniqueness and completeness ----------
expected_count = int((global_max - global_min) // ONE_SECOND_NS + 1)
timestamp_seen = np.zeros(expected_count, dtype=bool)

valid_timestamp_count = 0
off_grid_count = 0
out_of_range_count = 0

with h5py.File(DATASET_PATH, "r") as f:
    timestamps_ds = f["DATA"]["axis1"]

    for start in range(0, total_rows, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, total_rows)
        chunk = timestamps_ds[start:end]

        relative = chunk - global_min
        on_grid = (relative % ONE_SECOND_NS) == 0
        offsets = relative // ONE_SECOND_NS
        in_range = (offsets >= 0) & (offsets < expected_count)
        valid = on_grid & in_range

        off_grid_count += int((~on_grid).sum())
        out_of_range_count += int((~in_range).sum())

        valid_offsets = offsets[valid].astype(np.int64)
        timestamp_seen[valid_offsets] = True
        valid_timestamp_count += len(valid_offsets)

unique_timestamp_count = int(timestamp_seen.sum())
duplicate_timestamp_count = valid_timestamp_count - unique_timestamp_count
missing_timestamp_count = expected_count - unique_timestamp_count

timestamp_summary = pd.Series({
    "true_start": start_time,
    "true_end": end_time,
    "actual_rows": total_rows,
    "expected_1sec_rows": expected_count,
    "unique_timestamps": unique_timestamp_count,
    "duplicate_timestamps": duplicate_timestamp_count,
    "missing_timestamps": missing_timestamp_count,
    "off_grid_timestamps": off_grid_count,
    "stored_order_backward_jumps": backward_jumps,
    "stored_order_forward_jumps": forward_jumps,
    "continuous_blocks": len(block_lengths),
}, name="timestamp structure")

block_summary = pd.Series({
    "minimum_block_rows": int(block_lengths.min()),
    "median_block_rows": float(np.median(block_lengths)),
    "maximum_block_rows": int(block_lengths.max()),
    "expected_seconds_per_day": 24 * 60 * 60,
}, name="stored continuous blocks")

display(timestamp_summary.to_frame("value"))
display(block_summary.to_frame("value"))


,value
true_start,2018-06-01 00:00:00+00:00
true_end,2019-09-01 00:00:00+00:00
actual_rows,39484801
expected_1sec_rows,39484801
unique_timestamps,39484801
duplicate_timestamps,0
missing_timestamps,0
off_grid_timestamps,0
stored_order_backward_jumps,239
stored_order_forward_jumps,217


,value
minimum_block_rows,86399.0
median_block_rows,86400.0
maximum_block_rows,86401.0
expected_seconds_per_day,86400.0


## Global variable availability and quality checks

A single sampled day can be misleading at one-second resolution. Instead of
drawing conclusions from one block, we scan the measurement matrix globally
in manageable chunks.

For each variable we record its minimum, maximum, and missing-value count. We
also quantify the behaviours that matter most for later modelling:

- availability of the two candidate targets;
- humidity outside its apparent fractional range;
- wind direction outside `[0, 360)`;
- negative irradiance or generation values;
- pressure outside a broad atmospheric screening range;
- non-zero availability of the source azimuth/elevation fields.

The `P_Gaia > 12 kW` count is retained only as an exploratory description of
the high wind-power tail, consistent with Notebook 03. It is **not** a
cleaning threshold.


In [3]:
column_index = {name: i for i, name in enumerate(columns)}
VALUE_CHUNK_SIZE = 500_000

mins = np.full(n_columns, np.inf)
maxs = np.full(n_columns, -np.inf)
missing_counts = np.zeros(n_columns, dtype=np.int64)

counts = {
    "solar_nonzero": 0,
    "gaia_nonzero": 0,
    "gaia_above_12_kW": 0,
    "humidity_outside_0_1": 0,
    "wind_direction_outside_0_360": 0,
    "negative_GHI": 0,
    "negative_POA": 0,
    "negative_solar_power": 0,
    "negative_wind_power": 0,
    "pressure_outside_800_1100": 0,
    "azimuth_nonzero": 0,
    "elevation_nonzero": 0,
}

with h5py.File(DATASET_PATH, "r") as f:
    values_ds = f["DATA"]["block0_values"]

    for start in range(0, n_rows, VALUE_CHUNK_SIZE):
        end = min(start + VALUE_CHUNK_SIZE, n_rows)
        chunk = values_ds[start:end, :]

        missing = np.isnan(chunk)
        missing_counts += missing.sum(axis=0)

        safe_chunk = np.where(missing, np.nan, chunk)
        mins = np.minimum(mins, np.nanmin(safe_chunk, axis=0))
        maxs = np.maximum(maxs, np.nanmax(safe_chunk, axis=0))

        solar = chunk[:, column_index["P_Solar[kW]"]]
        gaia = chunk[:, column_index["P_Gaia[kW]"]]
        humidity = chunk[:, column_index["HUMIDITY[%]"]]
        wind_dir = chunk[:, column_index["WIND_DIR[deg]"]]
        ghi = chunk[:, column_index["GHI[kW1m2]"]]
        poa = chunk[:, column_index["POA Irr[kW1m2]"]]
        pressure = chunk[:, column_index["Pressure[mbar]"]]
        azimuth = chunk[:, column_index["Azimuth[deg]"]]
        elevation = chunk[:, column_index["Elevation[deg]"]]

        counts["solar_nonzero"] += int(np.sum(solar != 0))
        counts["gaia_nonzero"] += int(np.sum(gaia != 0))
        counts["gaia_above_12_kW"] += int(np.sum(gaia > 12))
        counts["humidity_outside_0_1"] += int(np.sum((humidity < 0) | (humidity > 1)))
        counts["wind_direction_outside_0_360"] += int(np.sum((wind_dir < 0) | (wind_dir >= 360)))
        counts["negative_GHI"] += int(np.sum(ghi < 0))
        counts["negative_POA"] += int(np.sum(poa < 0))
        counts["negative_solar_power"] += int(np.sum(solar < 0))
        counts["negative_wind_power"] += int(np.sum(gaia < 0))
        counts["pressure_outside_800_1100"] += int(np.sum((pressure < 800) | (pressure > 1100)))
        counts["azimuth_nonzero"] += int(np.sum(azimuth != 0))
        counts["elevation_nonzero"] += int(np.sum(elevation != 0))

variable_ranges = pd.DataFrame({
    "min": mins,
    "max": maxs,
    "missing": missing_counts,
}, index=columns)

availability = pd.DataFrame({
    "nonzero_observations": [
        counts["solar_nonzero"],
        counts["gaia_nonzero"],
        counts["azimuth_nonzero"],
        counts["elevation_nonzero"],
    ],
}, index=[
    "P_Solar[kW]",
    "P_Gaia[kW]",
    "Azimuth[deg]",
    "Elevation[deg]",
])
availability["nonzero_fraction"] = availability["nonzero_observations"] / n_rows

quality_flags = pd.Series({
    "humidity <0 or >1": counts["humidity_outside_0_1"],
    "wind direction outside [0,360)": counts["wind_direction_outside_0_360"],
    "negative GHI": counts["negative_GHI"],
    "negative POA irradiance": counts["negative_POA"],
    "negative solar power": counts["negative_solar_power"],
    "negative wind power": counts["negative_wind_power"],
    "pressure outside 800–1100 mbar": counts["pressure_outside_800_1100"],
    "Gaia power >12 kW (diagnostic only)": counts["gaia_above_12_kW"],
}, name="observations")

display(variable_ranges)
display(availability)
display(quality_flags.to_frame())


,min,max,missing
TEMPERATURE[degC],-40.100000,32.400000,0
HUMIDITY[%],0.000000,5.000000,0
WIND_SPEED[m1s],0.000000,29.000000,0
WIND_DIR[deg],0.000000,712.000000,0
GHI[kW1m2],0.000000,1.401000,0
POA Irr[kW1m2],0.000000,31.446000,0
P_Gaia[kW],0.000000,12.313000,0
P_Solar[kW],0.000000,9.831133,0
Pressure[mbar],991.599976,3000.000000,0
Azimuth[deg],-65.087997,55.719002,0


,nonzero_observations,nonzero_fraction
P_Solar[kW],22726233,0.575569
P_Gaia[kW],134890,0.003416
Azimuth[deg],32600,0.000826
Elevation[deg],26880,0.000681


,observations
humidity <0 or >1,86402
"wind direction outside [0,360)",1343
negative GHI,0
negative POA irradiance,0
negative solar power,0
negative wind power,0
pressure outside 800–1100 mbar,86855
Gaia power >12 kW (diagnostic only),22


## Interpretation

The 1-second file should not automatically be considered the most useful
resolution simply because it has the finest sampling interval.

### Timestamp structure

The complete one-second timeline is present:

- **39,484,801 observed timestamps**;
- **39,484,801 expected timestamps**;
- **0 missing timestamps**;
- **0 duplicate timestamps**;
- **0 off-grid timestamps**.

The unusual feature is the **stored row order**, not the temporal coverage.
The file contains **457 continuous blocks**, with a median block length of
86,400 observations — approximately one day at one-second resolution. Between
these blocks, the stored order contains 239 backward jumps and 217 forward
jumps.

Therefore, chronological forecasting sequences must be constructed from the
timestamps rather than from raw HDF5 row position.

### Variable representation

The variables are not represented equally densely at one-second resolution.

- `P_Solar[kW]` is non-zero in **22,726,233 observations
  (57.56%)**.
- `P_Gaia[kW]` is non-zero in only **134,890 observations
  (0.34%)**.
- `Azimuth[deg]` is non-zero in only **32,600 observations
  (0.083%)**.
- `Elevation[deg]` is non-zero in only **26,880 observations
  (0.068%)**.

These are actual zero values rather than missing (`NaN`) entries. Their very
different prevalence across variables shows that the one-second file should
not be assumed to contain the same effective information as the coarser
resolution files.

### Values requiring further investigation

Several source values are unusual enough to justify explicit
cross-resolution comparison:

- temperature reaches **−40.1 °C**;
- humidity reaches **5.0**, with **86,402 observations** outside the apparent
  fractional range `[0, 1]`;
- wind direction reaches **712°**, with **1,343 observations** outside
  `[0°, 360°)`;
- pressure reaches **3000 mbar**, with **86,855 observations** outside
  800–1100 mbar;
- POA irradiance reaches **31.446 kW/m²**, far above the rest of the scale and
  therefore requiring investigation;
- only **22 Gaia observations** exceed 12 kW.

These checks identify observations for investigation only. They do not yet
define cleaning rules.


## Findings

The 1-second SOLETE file contains **39,484,801 observations**, 11 variables,
and occupies approximately **3.53 GB**. Chunked HDF5 processing is therefore
more appropriate than loading the complete measurement matrix into pandas.

### 1. The time axis is complete, but the storage order is shuffled

The entire expected one-second grid from 1 June 2018 to 1 September 2019 is
present exactly once. There are no missing, duplicated, or off-grid
timestamps.

However, the HDF5 rows are stored as approximately daily continuous blocks
that are not arranged chronologically. This means row position cannot be used
as a substitute for time order when constructing lags or forecasting
sequences.

### 2. Finer resolution does not automatically mean richer modelling data

Solar power is densely represented, but Gaia wind power is non-zero in only
about **0.34%** of one-second observations. The source azimuth and elevation
fields are also almost entirely zero at this resolution.

This contrasts with the behaviour observed in the coarser files and indicates
that the published SOLETE resolutions are not simply interchangeable copies
of the same multivariable table sampled at different frequencies.

### 3. Several source-level anomalies are visible at one-second resolution

The one-second data contain unusual pressure, humidity, wind-direction,
temperature, and POA-irradiance values. Because these patterns are visible in
the high-resolution source file, later preprocessing decisions should be based
on cross-resolution evidence rather than assuming that they were introduced
only by downsampling.

No observation is corrected or removed here.

### 4. Practical implication for resolution selection

The one-second file provides valuable diagnostic detail, but it also introduces
substantial computational cost while some important variables are extremely
sparse or represented differently from the coarser files.

Therefore, the modelling resolution should be selected using several criteria
together:

- temporal completeness;
- availability of the forecasting targets;
- consistency of predictor variables;
- behaviour of identified anomalies;
- compatibility with the intended forecast horizons; and
- computational practicality.

## Next step

The next notebook will compare the **60-minute, 5-minute, 1-minute, and
1-second** SOLETE files directly.

That comparison will allow the modelling resolution and any later
preprocessing rules to be justified from the observed data rather than chosen
in advance.
